# **Pull GitHub Repository**

In [ ]:
!pip install -q torchmetrics timm

In [ ]:
from google.colab import userdata

!rm -rf /content/ECM3401_Individual_Project

token = userdata.get('GitHub')
!git clone -b swin_transformer_layer https://{token}@github.com/sccthomas/ECM3401_Individual_Project.git

In [ ]:
import sys

sys.path.append('/content/ECM3401_Individual_Project/')
!ls /content/ECM3401_Individual_Project/src/

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# **Define the Model**

In [ ]:
import torch
from src.vision_transformer.model import SemanticSegmentationVisionTransformer
from src.self_supervised_learning.contrastive_loss import ContrastivePreTraining

# --------------------------------------------
# Parameters
# --------------------------------------------
device = torch.device("cuda")

image_dims = (3, 256, 256)  # Input image dimensions
patch_embedding_scale_1 = (16, 768)  # Patch size and embedding dimension for scale 1
patch_embedding_scale_2 = (8, 512)  # Patch size and embedding dimension for scale 2
patch_embedding_scale_3 = (4, 256)  # Patch size and embedding dimension for scale 3

# --------------------------------------------
# Model Initialization
# --------------------------------------------
model = SemanticSegmentationVisionTransformer(
    # - Image dimensions
    image_dims=image_dims,
    # - Hyper Parameters
    num_encoder_layers=6,
    use_heavyweight_decoder=False,
    use_swin_transformer=False,
    use_skip_layer_gated_attention=False,
    use_learnable_skip_layers=True,
    skip_layer_ratio=2,
    encoder_dropout_rate=0.15,
    patch_fusion_dropout_rate=0.25,
    decoder_dropout_rate=0.25,
    num_encoder_heads=4,
    num_classes=1,
    # - Scales
    patch_embedding_scale_1=patch_embedding_scale_1,
    patch_embedding_scale_2=patch_embedding_scale_2,
    patch_embedding_scale_3=patch_embedding_scale_3,
).to(device)

# ssl_model = MaskedRegionLoss(
#     model=model,
#     max_patch_size=32,
# ).to(device)

ssl_model = ContrastivePreTraining(
    model=model,
    encoder_dims=[768, 512, 256],
    projection_dim=128,
).to(device)

# **Train the Model**

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from src.dataset.snow import SnowDataset
from src.training.self_supervised_learning import train_model

# --------------------------------------------
# Parameters
# --------------------------------------------
dataset_dir = "/content/drive/MyDrive/snow_dataset"  # Replace with your dataset path
batch_size = 50
num_epochs = 50
learning_rate = 1e-4
patience = 5  # Early stopping patience

# --------------------------------------------
# Dataset and DataLoader
# --------------------------------------------

# Load the dataset and split it into train and validation sets
train_dataset = SnowDataset(
    dataset_dir_path=dataset_dir,
    len_override=10_000,
    normalize=True,
    resize=True,
)
print("Length of dataset:", len(train_dataset))
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=12,
    persistent_workers=True,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=12,
    persistent_workers=True,
    pin_memory=True,
)
# --------------------------------------------
# Loss, Optimizer, and Scheduler
# --------------------------------------------
optimizer = optim.AdamW(ssl_model.parameters(), lr=learning_rate, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=1)

# --------------------------------------------
# Mixed Precision Setup
# --------------------------------------------
scaler = torch.amp.GradScaler(device.type)

# --------------------------------------------
# Train Model
# --------------------------------------------
train_model(
    ssl_model=ssl_model,
    num_epochs=num_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    train_loader=train_loader,
    val_loader=val_loader,
    patience=patience,
    device=device,
)

In [ ]:
!cp /content/best_model_ssl.pth /content/drive/MyDrive/best_model_ssl_scale_3_contrastive.pth

# **Evaluate the Model**

In [ ]:
import torch

# Load the model's state_dict
# path = 'best_model.pth'
# model.load_state_dict(torch.load(path))
# model = model.eval()
# Load the SSL model's state_dict
path = '/content/drive/MyDrive/best_model_ssl_scale_3_contrastive.pth'
ssl_model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
ssl_model = ssl_model.eval()

In [ ]:
from src.dataset.snow import SnowDataset
from torch.utils.data import DataLoader

dataset_dir = "/content/drive/MyDrive/snow_dataset"

# Dataset and DataLoader
batch_size = 10

train_dataset = SnowDataset(
    dataset_dir_path=dataset_dir,
    len_override=30,
    normalize=True,
)
validation_dataset = SnowDataset(
    dataset_dir_path=dataset_dir,
    len_override=30,
    normalize=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=1,
)
val_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=1,
)

In [ ]:
images_original, _ = next(iter(val_loader))
images, masks = next(iter(train_loader))
images, masks = images.to(device), masks.to(device)
outputs = ssl_model.model(images)

## **Contrastive SSL**

In [ ]:
from src.self_supervised_learning.contrastive_loss import visualize_tsne

visualize_tsne(ssl_model, images)

In [ ]:
from src.training.visualisation import display_attention_weights

image_idx = 0

display_attention_weights(
    model=ssl_model.model,
    img_original=images_original[image_idx],
    img_pre=images[image_idx],
    patch_sizes=[32, 16, 8],
    use_max_pooling=False,
)

## **Generative SSL**

In [ ]:
import torch.nn as nn

criterion = nn.BCEWithLogitsLoss()  # Binary cross-entropy loss with logits
loss = criterion(outputs, masks)
print(loss)
del loss
del criterion

In [ ]:
from src.training.visualisation import display_attention_weights, display_tensor_mask, display_tensor_image

outputs = torch.sigmoid(outputs)
image_idx = 0

In [ ]:
display_tensor_image(images_original[image_idx])

In [ ]:
display_tensor_mask(masks[image_idx])

In [ ]:
display_tensor_mask(outputs[image_idx] > 0.5)

In [ ]:
display_attention_weights(
    model=ssl_model.model,
    img_original=images_original[image_idx],
    img_pre=images[image_idx],
    patch_sizes=[32, 16, 8],
    use_max_pooling=False,
)